# Fase 3 · Integridad y calidad de los datos  (RA3 · UD3)

**Reto (auditoría técnica):** un error en los datos del puerto puede provocar una mala planificación de atraques
o una factura equivocada. Os encargan **auditar la integridad** de la plataforma:

| Parte | Qué hacéis | CE |
|---|---|---|
| A | Perfilar los datos: ¿qué defectos tienen? | 3.a |
| B | Reglas de calidad, cuarentena y zona *silver* | 3.a, 3.b |
| C | Sumas de verificación en HDFS y de extremo a extremo | 3.c |
| D | Corrupción controlada de un bloque: qué hace cada servidor | 3.c, 3.d |
| E | ¿Cuánto cuesta la integridad? Medir la sobrecarga | 3.b |
| F | Informe de auditoría | 3.a–3.d |

Requisito previo: zonas *bronze* de la fase 1 (AIS, escalas, sensores, contenedores).

In [ ]:
from pyspark.sql import functions as F, Window
from puerto.sesion import crear_spark, cronometro, sello
PAREJA = "00"   # ✏️ vuestro número de pareja: su salida debe verse en las capturas de la memoria
sello(PAREJA)
from puerto import hdfs_web as h, calidad as q
from puerto.validacion import registrar_udfs, contenedor_valido, imo_valido
import pandas as pd, matplotlib.pyplot as plt, os
os.makedirs("/workspace/entregas/F3", exist_ok=True)
spark = crear_spark("F3-integridad")
registrar_udfs(spark)
cont_valido = F.udf(contenedor_valido, "boolean")
imo_ok = F.udf(imo_valido, "boolean")

## A · Perfilado

In [ ]:
escalas = spark.read.parquet("/puerto/bronze/escalas")
q.perfil(escalas).show(30, truncate=False)
escalas.describe("toneladas", "teu").show()
print("Filas:", escalas.count(), "| id_escala distintos:", escalas.select("id_escala").distinct().count())

**✏️ TAREA A1.** Perfilad también **contenedores**, **sensores** y **AIS**. Para cada tabla, listad en la memoria los
defectos que encontréis, clasificados por **dimensión de calidad** (completitud, validez, exactitud, consistencia,
unicidad, actualidad) y con una estimación de su impacto en el negocio.

In [ ]:
# ✏️ TAREA A1

## B · Reglas de calidad y cuarentena
Cada regla describe lo que una fila **correcta** cumple. Las reglas **críticas** mandan la fila a cuarentena;
las demás solo se informan (o se corrigen). Leed `src/puerto/calidad.py`.

In [ ]:
reglas_escalas = [
    q.completitud("id_muelle", critica=True),
    q.completitud("eta", critica=True),
    q.completitud("ata", critica=True),
    q.rango("toneladas", 0, 500_000, critica=True),
    q.unicidad(["id_escala"], critica=True),
    q.consistencia("atd_posterior_ata", F.col("atd") >= F.col("ata"), critica=True,
                   descripcion="la salida no puede ser anterior a la llegada"),
    q.personalizada("imo_valido", "validez", imo_ok("imo"), descripcion="dígito de control IMO"),
    q.consistencia("nombre_buque_limpio", F.col("buque") == F.trim(F.col("buque")),
                   descripcion="nombre sin espacios sobrantes"),
]
marcado = q.evaluar(escalas, reglas_escalas).cache()
inf = q.informe(marcado, reglas_escalas)
inf.show(truncate=False)
print("Índice de calidad de escalas:", q.puntuacion(inf))

In [ ]:
validas, cuarentena = q.separar(marcado, reglas_escalas)
# Correcciones NO críticas: normalizar el nombre del buque
validas = validas.withColumn("buque", F.initcap(F.trim("buque")))
validas.write.mode("overwrite").partitionBy("anio").parquet("/puerto/silver/escalas")
cuarentena.write.mode("overwrite").parquet("/puerto/cuarentena/escalas")
inf.write.mode("overwrite").parquet("/puerto/control/calidad/escalas")
inf.toPandas().to_csv("/workspace/entregas/F3/calidad_escalas.csv", index=False)
print("válidas:", validas.count(), "| cuarentena:", cuarentena.count())
cuarentena.groupBy("_motivo").count().orderBy(F.desc("count")).show(truncate=False)

**✏️ TAREA B1.** Las toneladas registradas en **kg** (valores ×1000) hoy van a cuarentena. Proponed e implementad una
regla de **corrección** en lugar de descarte y justificad cuándo es preferible corregir y cuándo descartar.

**✏️ TAREA B2.** Implementad reglas y zona *silver* para **contenedores** (dígito de control ISO 6346 con `cont_valido`,
peso bruto entre 1 000 y 36 000 kg, reefer `45R1` con temperatura de consigna, movimiento único) y para **sensores**:
- valor centinela `-999`;
- temperatura en °F (convertid a °C);
- lecturas duplicadas;
- sensor «congelado»: desviación típica 0 en las últimas 120 lecturas (pista en la celda siguiente).

Guardad los informes en `/puerto/control/calidad/<tabla>` y en `entregas/F3/`.

In [ ]:
# Pista para detectar un sensor «congelado» (exactitud): misma lectura durante mucho tiempo
sensores = spark.read.parquet("/puerto/bronze/sensores")
w = Window.partitionBy("id_sensor", "variable").orderBy("ts").rowsBetween(-119, 0)
congelado = (sensores.filter("variable = 'so2'")
             .withColumn("desv_2h", F.stddev("valor").over(w))
             .filter("desv_2h = 0"))
congelado.groupBy("id_sensor", F.to_date("ts").alias("dia")).count().orderBy("dia").show()

In [ ]:
# ✏️ TAREA B2

## C · Sumas de verificación (checksums)
HDFS guarda, junto a cada bloque, un fichero `.meta` con un **CRC32C por cada 512 bytes**. El cliente las calcula al
escribir y las comprueba al leer. La suma de un fichero completo que devuelve HDFS es un «MD5 de MD5 de CRC»:

In [ ]:
print(h.checksum("/puerto/raw/contenedores/movimientos_2025.csv"))

Comparad el checksum del **mismo contenido** con otro tamaño de bloque (en PowerShell):
```powershell
docker compose exec namenode hdfs dfs -D dfs.blocksize=8388608 -cp -f /puerto/raw/contenedores/movimientos_2025.csv /puerto/experimentos/mov_8mb.csv
.\bda.ps1 hdfs dfs -checksum /puerto/raw/contenedores/movimientos_2025.csv /puerto/experimentos/mov_8mb.csv
docker compose exec namenode hdfs dfs -D dfs.checksum.combine.mode=COMPOSITE_CRC -checksum /puerto/raw/contenedores/movimientos_2025.csv /puerto/experimentos/mov_8mb.csv
```
Y la verificación **de extremo a extremo** con SHA-256 (del fichero generado al fichero en HDFS):
```powershell
.\bda.ps1 verificar
```
**✏️ TAREA C1.** ¿Por qué el checksum por defecto cambia con el tamaño de bloque y el `COMPOSITE_CRC` no?
¿Qué protege el CRC de HDFS y qué protege el manifiesto SHA-256? ¿Por qué no se usa SHA-256 para cada trozo de 512 bytes?
Comparad CRC32C, MD5 y SHA-256 (velocidad, detección de errores, resistencia a manipulaciones).

## D · Corrupción controlada: ¿qué hace cada servidor?
**Seguid los pasos en PowerShell** y guardad la salida de cada uno:

1. Copia de trabajo (¡nunca experimentéis con los originales!):
   `.\bda.ps1 hdfs dfs -cp -f /puerto/raw/contenedores/movimientos_2025.csv /puerto/experimentos/victima.csv`
2. Bloques y réplicas: `docker compose exec namenode bash /scripts/localizar_bloques.sh /puerto/experimentos/victima.csv`
   Anotad el identificador del primer bloque (`blk_10737...`, sin el sufijo `_1xxx`).
3. Corromped UNA réplica (si el nodo no la tiene, probad con el -2 o el -3):
   `docker exec bda-puerto-datanode-1 bash /scripts/corromper_bloque.sh blk_XXXXXXXXXX`
4. Verificación **en el propio DataNode** (sin leer desde un cliente):
   `docker exec bda-puerto-datanode-1 bash -c 'B=$(find /tmp/hdfs/data -type f -name blk_XXXXXXXXXX); hdfs debug verifyMeta -meta $(ls ${B}_*.meta) -block $B'`
5. Leed el fichero como cliente: `.\bda.ps1 hdfs dfs -cat /puerto/experimentos/victima.csv > $null`
   Buscad en la salida `ChecksumException` / `Found Checksum error`: ¿se interrumpe la lectura?
6. `.\bda.ps1 hdfs fsck /puerto/experimentos/victima.csv -files -blocks -locations` y
   `.\bda.ps1 hdfs fsck / -list-corruptfileblocks`. Repetidlo al cabo de 1-2 minutos: ¿qué ha hecho el NameNode?
7. (Opcional, arriesgado) Corromped las **tres** réplicas del bloque. ¿Qué ocurre ahora al leer? ¿Cómo se recupera?

In [ ]:
# Estado visto desde Python (repetid tras cada paso)
print(h.fsck("/puerto/experimentos/victima.csv")[-1200:])

**✏️ TAREA D1.** Haced un **diagrama de secuencia** del *pipeline* de escritura (cliente → DN1 → DN2 → DN3 y
confirmaciones) indicando **quién calcula y quién comprueba** los checksums, y otro de la lectura con la réplica
corrupta. Explicad el papel de: el cliente, cada DataNode (verificación al recibir, *block scanner* cada
`dfs.datanode.scan.period.hours`), y el NameNode (marca la réplica corrupta, ordena re-replicar, borra la mala).

## E · ¿Cuánto cuesta la integridad?
Escribimos el mismo conjunto de datos variando **réplicas** y **tipo de checksum** y medimos tiempo y espacio.

In [ ]:
datos = spark.read.parquet("/puerto/bronze/ais").cache()
datos.count()
conf = spark.sparkContext._jsc.hadoopConfiguration()
filas = []
for replicas in [1, 2, 3]:
    for tipo in ["CRC32C", "NULL"]:
        conf.set("dfs.replication", str(replicas)); conf.set("dfs.checksum.type", tipo)
        ruta = f"/puerto/experimentos/sobrecarga_r{replicas}_{tipo}"
        t = {}
        try:
            with cronometro(f"r{replicas}-{tipo}", t):
                datos.write.mode("overwrite").parquet(ruta)
        except Exception as e:                   # anotad el error si vuestra versión no admite la combinación
            print("ERROR", replicas, tipo, str(e)[:200]); continue
        r = h.resumen(ruta)
        filas.append({"replicas": replicas, "checksum": tipo, "escritura_s": list(t.values())[0],
                      "MB_logicos": round(r["bytes"] / 2**20, 1), "MB_en_disco": round(r["bytes_en_disco"] / 2**20, 1)})
conf.set("dfs.replication", "3"); conf.set("dfs.checksum.type", "CRC32C")
sobrecarga = pd.DataFrame(filas)
sobrecarga.to_csv("/workspace/entregas/F3/sobrecarga.csv", index=False)
sobrecarga

In [ ]:
t = {}
with cronometro("sin reglas", t):
    spark.read.parquet("/puerto/bronze/escalas").write.mode("overwrite").format("noop").save()
with cronometro("con reglas", t):
    q.evaluar(spark.read.parquet("/puerto/bronze/escalas"), reglas_escalas).write.mode("overwrite").format("noop").save()
print(f"Sobrecarga de la validación: {100 * (t['con reglas'] / t['sin reglas'] - 1):.0f} %")

Benchmark estándar de E/S de HDFS (opcional, tarda unos minutos):
```powershell
docker compose exec namenode bash -c "hadoop jar /opt/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-client-jobclient-*-tests.jar TestDFSIO -write -nrFiles 4 -size 64MB -resFile /tmp/dfsio.txt; cat /tmp/dfsio.txt"
```
**✏️ TAREA E1.** Con vuestros números: ¿cuánto espacio y tiempo cuesta pasar de 1 a 3 réplicas? ¿Y los checksums?
Relacionadlo con el **CE3.b**: a más volumen, más probabilidad de corrupción silenciosa (*bit rot*), así que la
sobrecarga es un «seguro». Calculad cuántos errores de lectura no recuperables cabría esperar al leer 1 PB con discos
de tasa UBER = 1 error cada 10¹⁵ bits.

## F · Informe de auditoría
Completad `entregas/F3/MEMORIA_F3.md` siguiendo la plantilla de auditoría: alcance, metodología, hallazgos
(con severidad y evidencia), riesgos, recomendaciones y plan de acción.

In [ ]:
spark.stop()